In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("Working dir:", os.getcwd())

ModuleNotFoundError: No module named 'google.colab'

In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

# Adjust path below to where your step7_pca.csv is located in your Drive
RAW_PATH = "/content/drive/MyDrive/Colab Notebooks/data/processed/Dataset7.csv"
if not os.path.exists(RAW_PATH):
    # fallback: try current folder
    RAW_PATH = "Dataset7.csv"

df = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH, "shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


NameError: name 'os' is not defined

In [ ]:
# Cell 2: Prepare features and target (shared)
from sklearn.model_selection import train_test_split

# Ensure target exists
assert 'selling_price' in df.columns, "selling_price not in dataset"

X = df.drop(columns=['selling_price'])
y = df['selling_price']

# If any NaNs remain, fill with median for modeling convenience
X = X.fillna(X.median())

# Train-test split (same for all members to ensure fair comparison)
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error

ks = [3, 5, 10, 20]
knn_results = []
for k in ks:
    knn = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train, y_train)
    yp = knn.predict(X_test)
    knn_results.append((k, r2_score(y_test, yp), mean_absolute_error(y_test, yp)))

for k, r2v, mae in knn_results:
    print(f"k={k} -> R2: {r2v:.3f}, MAE: {mae:.0f}")

plt.figure(figsize=(6,4))
plt.plot([r[0] for r in knn_results], [r[1] for r in knn_results], marker='o')
plt.xlabel("k (neighbors)")
plt.ylabel("R² Score")
plt.title("KNN: k vs R²")
plt.grid(True)
plt.show()

# Choose best k and show Actual vs Predicted
best_k = max(knn_results, key=lambda t: t[1])[0]
knn_best = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)
knn_best.fit(X_train, y_train)
y_pred_knn = knn_best.predict(X_test)

plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred_knn, alpha=0.5, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title(f"KNN (k={best_k}) Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

print("KNN is a non-parametric method; low k can overfit while high k smooths predictions. Choose k by cross-validation.")